<a href="https://colab.research.google.com/github/Shineii86/Quickdraw/blob/main/notebooks/Quickdraw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center">
  <img src="https://raw.githubusercontent.com/Shineii86/Quickdraw/main/images/Quickdraw.png" width="300px" alt="Quickdraw Achievement">
  <h1>🤠 Quickdraw Achievement Helper</h1>
  <p><b>Automate opening and immediately closing an issue to unlock the Quickdraw badge — all from your browser.</b></p>
</div>

---

> ⚠️ **IMPORTANT**
> - This script uses the GitHub REST API to create and then immediately close an issue.
> - **Quickdraw requires closing an issue or pull request within 5 minutes of opening it.** This tool automates the process in just a few seconds.
> - **Use responsibly.** Artificially inflating achievements may be viewed negatively by potential employers or collaborators, and may violate GitHub's Terms of Service.
> - You need a **Personal Access Token (classic)** with `repo` scope for your account.
> - The script will create a public repository (or use an existing one), open an issue, and close it immediately.

---

In [ ]:
#@title 📦 1. Install Dependencies & Setup
!pip install -q requests

import os
import json
import time
import random
import base64
import requests
from datetime import datetime

print("✅ All dependencies installed and imported.")

In [ ]:
#@title ⚙️ 2. Configuration & Run

# =============================================================================
#  🔧 Configuration
# =============================================================================

# Your main GitHub username (the one that will earn the achievement)
USERNAME = "Shineii86"  #@param {type:"string"}

# Your Personal Access Token (classic) with 'repo' scope
TOKEN = "ghp_xxxxxxxxxxxxxxxxxxxx"  #@param {type:"string"}

# Repository name (will be created if it does not exist)
REPO_NAME = "quickdraw-demo"  #@param {type:"string"}

# Type of item to create and close: "issue" or "pull_request" (both work for Quickdraw)
ITEM_TYPE = "issue"  #@param ["issue", "pull_request"] {type:"string"}

# Delay before closing (seconds) – keep under 300 (5 minutes) for Quickdraw
CLOSE_DELAY = 2  #@param {type:"number"}

# =============================================================================
#  🤠 Quickdraw Automation Script
# =============================================================================

API_BASE = "https://api.github.com"
HEADERS = {
    "Authorization": f"Bearer {TOKEN}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28"
}

def api_request(method, endpoint, data=None):
    """Make a GitHub REST API request."""
    url = f"{API_BASE}{endpoint}"
    response = requests.request(method, url, headers=HEADERS, json=data)
    if response.status_code not in [200, 201, 204]:
        raise Exception(f"API request failed: {response.status_code} - {response.text}")
    return response.json() if response.text else {}

def create_repo(repo_name):
    """Create a new public repository."""
    print(f"📦 Creating repository '{repo_name}'...")
    data = {
        "name": repo_name,
        "private": False,
        "auto_init": True
    }
    result = api_request("POST", "/user/repos", data)
    print(f"   ✅ Repository created: {result['html_url']}")
    return result

def get_file_sha(repo_name, path):
    """Get the SHA of a file in the repository."""
    endpoint = f"/repos/{USERNAME}/{repo_name}/contents/{path}"
    try:
        result = api_request("GET", endpoint)
        return result.get("sha")
    except:
        return None

def create_or_update_file(repo_name, path, content, message, branch="main"):
    """Create or update a file in the repository."""
    print(f"📝 Updating {path} on branch '{branch}'...")
    data = {
        "message": message,
        "content": base64.b64encode(content.encode()).decode(),
        "branch": branch
    }
    sha = get_file_sha(repo_name, path)
    if sha:
        data["sha"] = sha
    endpoint = f"/repos/{USERNAME}/{repo_name}/contents/{path}"
    result = api_request("PUT", endpoint, data)
    print(f"   ✅ File updated")
    return result

def get_main_sha(repo_name):
    """Get the SHA of the main branch."""
    endpoint = f"/repos/{USERNAME}/{repo_name}/git/refs/heads/main"
    result = api_request("GET", endpoint)
    return result["object"]["sha"]

def create_branch(repo_name, branch_name, base_sha):
    """Create a new branch."""
    print(f"🌿 Creating branch '{branch_name}'...")
    endpoint = f"/repos/{USERNAME}/{repo_name}/git/refs"
    data = {
        "ref": f"refs/heads/{branch_name}",
        "sha": base_sha
    }
    result = api_request("POST", endpoint, data)
    print(f"   ✅ Branch created")
    return result

def create_pull_request(repo_name, head_branch, title, body):
    """Create a pull request."""
    print(f"🔀 Creating pull request...")
    endpoint = f"/repos/{USERNAME}/{repo_name}/pulls"
    data = {
        "title": title,
        "body": body,
        "head": head_branch,
        "base": "main"
    }
    result = api_request("POST", endpoint, data)
    print(f"   ✅ PR created: {result['html_url']}")
    return result

def create_issue(repo_name, title, body):
    """Create a new issue."""
    print(f"📝 Creating issue...")
    endpoint = f"/repos/{USERNAME}/{repo_name}/issues"
    data = {
        "title": title,
        "body": body
    }
    result = api_request("POST", endpoint, data)
    print(f"   ✅ Issue created: {result['html_url']}")
    return result

def close_issue(repo_name, issue_number):
    """Close an issue."""
    print(f"🔒 Closing issue #{issue_number}...")
    endpoint = f"/repos/{USERNAME}/{repo_name}/issues/{issue_number}"
    data = {"state": "closed"}
    result = api_request("PATCH", endpoint, data)
    print(f"   ✅ Issue closed")
    return result

def close_pull_request(repo_name, pr_number):
    """Close a pull request."""
    print(f"🔒 Closing pull request #{pr_number}...")
    endpoint = f"/repos/{USERNAME}/{repo_name}/pulls/{pr_number}"
    data = {"state": "closed"}
    result = api_request("PATCH", endpoint, data)
    print(f"   ✅ Pull request closed")
    return result

# =============================================================================
#  🚀 Main Execution
# =============================================================================

print(f"\n🤠 Quickdraw Achievement Helper for user '{USERNAME}'")
print(f"Repository: {USERNAME}/{REPO_NAME}")
print(f"Item type: {ITEM_TYPE}")
print(f"Close delay: {CLOSE_DELAY} seconds\n")

# Step 1: Create or use repository
try:
    repo = create_repo(REPO_NAME)
    # Wait a moment for repository initialization
    time.sleep(3)
except Exception as e:
    if "already exists" in str(e):
        print(f"   ℹ️ Repository already exists, using existing one.")
    else:
        raise

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

if ITEM_TYPE == "issue":
    # Create an issue
    issue_title = f"🤠 Quickdraw Test - {timestamp}"
    issue_body = "This issue was created automatically to earn the Quickdraw achievement.\n\nIt will be closed within 5 minutes.\n\n*Quickdraw = Close an issue or PR within 5 minutes of opening.*"
    issue = create_issue(REPO_NAME, issue_title, issue_body)
    issue_number = issue["number"]
    
    # Wait a moment
    print(f"⏳ Waiting {CLOSE_DELAY} seconds before closing...")
    time.sleep(CLOSE_DELAY)
    
    # Close the issue
    close_issue(REPO_NAME, issue_number)
    
else:  # pull_request
    # Get main SHA
    main_sha = get_main_sha(REPO_NAME)
    
    # Create a new branch
    branch_name = f"quickdraw-{datetime.now().strftime('%Y%m%d%H%M%S')}"
    create_branch(REPO_NAME, branch_name, main_sha)
    time.sleep(2)
    
    # Make a small change on the branch
    file_content = f"# Quickdraw Achievement Test\n\nThis file was created to trigger the Quickdraw achievement.\n\nTimestamp: {timestamp}\n"
    create_or_update_file(
        REPO_NAME,
        "quickdraw-test.md",
        file_content,
        f"Add quickdraw-test.md for Quickdraw achievement",
        branch_name
    )
    time.sleep(2)
    
    # Create a pull request
    pr_title = f"🤠 Quickdraw Test - {timestamp}"
    pr_body = "This PR was created automatically to earn the Quickdraw achievement.\n\nIt will be closed within 5 minutes.\n\n*Quickdraw = Close an issue or PR within 5 minutes of opening.*"
    pr = create_pull_request(REPO_NAME, branch_name, pr_title, pr_body)
    pr_number = pr["number"]
    
    # Wait a moment
    print(f"⏳ Waiting {CLOSE_DELAY} seconds before closing...")
    time.sleep(CLOSE_DELAY)
    
    # Close the pull request
    close_pull_request(REPO_NAME, pr_number)

print("\n" + "="*50)
print(f"✨ Success! {ITEM_TYPE.replace('_', ' ').title()} opened and closed within {CLOSE_DELAY} seconds.")
print(f"📊 Check your profile: https://github.com/{USERNAME}")
print("\n🔔 Note: The Quickdraw achievement may take a few minutes to appear.")
print("   You can also check: https://github.com/settings/achievements")
print("\n---")
print("🤠 Quickdraw Helper By [Shinei Nouzen](https://github.com/Shineii86/Quickdraw)")


---

### 📚 How It Works

- The script creates a public repository (or uses an existing one).
- It opens either an **issue** or a **pull request** (you choose which).
- After a short delay (2 seconds by default), it immediately closes that issue or PR.
- This satisfies the Quickdraw requirement: "Close an issue or pull request within 5 minutes of opening."

### ⚠️ Important Notes

- **The repository must be public** for the achievement to count.
- **You can use either an issue or a PR** — both work for Quickdraw.
- **The closing must happen within 5 minutes.** The script's default 2-second delay is well within this window.
- **Quickdraw is earned only once** — it does not have tiers.
- Achievements may take a few minutes to a few hours to appear on your profile.

---

<div align="center">

**Copyright [Shinei Nouzen](https://github.com/Shineii86) All Rights Reserved.**  
*Made with ❤️ for GitHub achievement enthusiasts*


***Found this useful? Give it a Star! ⭐ [Shineii86/Quickdraw](https://github.com/Shineii86/Quickdraw)***
</div>